# Preprocessing — Imputación de nulos

Columnas con nulos a tratar:
| Columna | Estrategia |
|---|---|
| `CARD_ISSUER` | Rellenar con `"Other"` |
| `CARD_TYPE` | Rellenar con `"Other"` |
| `CUSTO_TRANSACAO` | Mediana agrupada por `MERCHANT_NAME`, `FEE_DATE` (mes), `CARD_TYPE`, `CAMARA_COMPENSACION` |

## 1. Setup e imports

In [1]:
#from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.6f}".format)

DATA_DIR = Path("../data/raw/banorte_xb")
N_MONTHS = 4

## 2. Carga de datos

In [2]:
files = sorted(DATA_DIR.rglob("*.parquet"))[-N_MONTHS:]

dfs = []
for f in files:
    df_tmp = pd.read_parquet(f)
    year, month = f.parts[-2], f.stem
    df_tmp["year_month"] = f"{year}-{month}"
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

# Conversión de tipos necesarios para la imputación
df["CUSTO_TRANSACAO"] = pd.to_numeric(df["CUSTO_TRANSACAO"], errors="coerce")
df["FEE_DATE"] = pd.to_datetime(df["FEE_DATE"], errors="coerce")
df["fee_year_month"] = df["FEE_DATE"].dt.to_period("M")

print(f"Filas cargadas: {len(df):,}")

Filas cargadas: 5,868,722


## 3. Estado de nulos — antes de imputar

In [3]:
TARGET_COLS = ["CARD_ISSUER", "CARD_TYPE", "CUSTO_TRANSACAO"]

def null_report(dataframe: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    return pd.DataFrame({
        "null_count": dataframe[cols].isnull().sum(),
        "null_pct": (dataframe[cols].isnull().sum() / len(dataframe) * 100).round(4),
    })

print("ANTES:")
null_report(df, TARGET_COLS)

ANTES:


,null_count,null_pct
CARD_ISSUER,19809,0.337500
CARD_TYPE,15,0.000300
CUSTO_TRANSACAO,9117,0.155300


## 4. Imputar CARD_ISSUER y CARD_TYPE → `"Other"`

In [4]:
df["CARD_ISSUER"] = df["CARD_ISSUER"].fillna("Other")
df["CARD_TYPE"] = df["CARD_TYPE"].fillna("Other")

print("CARD_ISSUER — distribución después de imputar:")
print(df["CARD_ISSUER"].value_counts().head(10))
print()
print("CARD_TYPE — distribución después de imputar:")
print(df["CARD_TYPE"].value_counts())

CARD_ISSUER — distribución después de imputar:
CARD_ISSUER
BANCOMER         2873743
BANCOPPEL         519116
SANTANDER         441817
BANAMEX           381931
AZTECA            315742
BANORTE           267416
COMPROPAGO        213132
NUBANK            190114
HSBC              133906
MERCADO LIBRE     109506
Name: count, dtype: int64

CARD_TYPE — distribución después de imputar:
CARD_TYPE
DEBIT      4528417
CREDIT     1339832
PREPAID        303
COMBO          155
Other           15
Name: count, dtype: int64


## 5. Imputar CUSTO_TRANSACAO — mediana por grupo

Agrupamos por `MERCHANT_NAME`, `fee_year_month`, `CARD_TYPE`, `CAMARA_COMPENSACION` y usamos la mediana del grupo como costo aproximado.

Si el grupo completo no tiene ningún valor conocido (grupo 100% nulo), caemos al siguiente nivel de fallback hasta llegar a la mediana global.

In [5]:
GROUP_LEVELS = [
    ["MERCHANT_NAME", "fee_year_month", "CARD_TYPE", "CAMARA_COMPENSACION"],
    ["MERCHANT_NAME", "fee_year_month", "CAMARA_COMPENSACION"],
    ["MERCHANT_NAME", "CAMARA_COMPENSACION"],
    ["CAMARA_COMPENSACION"],
]

remaining_before = df["CUSTO_TRANSACAO"].isnull().sum()
print(f"Nulos antes: {remaining_before:,}")

for level, group_cols in enumerate(GROUP_LEVELS, start=1):
    mask = df["CUSTO_TRANSACAO"].isnull()
    if not mask.any():
        break
    group_median = df.groupby(group_cols)["CUSTO_TRANSACAO"].transform("median")
    df.loc[mask, "CUSTO_TRANSACAO"] = group_median[mask]
    remaining = df["CUSTO_TRANSACAO"].isnull().sum()
    print(f"  Nivel {level} ({', '.join(group_cols)}): nulos restantes = {remaining:,}")

# Fallback final: mediana global
if df["CUSTO_TRANSACAO"].isnull().any():
    global_median = df["CUSTO_TRANSACAO"].median()
    df["CUSTO_TRANSACAO"] = df["CUSTO_TRANSACAO"].fillna(global_median)
    print(f"  Fallback global (mediana={global_median:.6f}): nulos restantes = {df['CUSTO_TRANSACAO'].isnull().sum():,}")

print(f"\nNulos después: {df['CUSTO_TRANSACAO'].isnull().sum():,}")

Nulos antes: 9,117
  Nivel 1 (MERCHANT_NAME, fee_year_month, CARD_TYPE, CAMARA_COMPENSACION): nulos restantes = 0

Nulos después: 0


## 6. Verificación final — sin nulos en columnas tratadas

In [6]:
report_after = null_report(df, TARGET_COLS)
print("DESPUES:")
display(report_after)

assert report_after["null_count"].sum() == 0, "Aun hay nulos en las columnas tratadas!"
print("\nOK — sin nulos en CARD_ISSUER, CARD_TYPE y CUSTO_TRANSACAO.")

DESPUES:


,null_count,null_pct
CARD_ISSUER,0,0.000000
CARD_TYPE,0,0.000000
CUSTO_TRANSACAO,0,0.000000



OK — sin nulos en CARD_ISSUER, CARD_TYPE y CUSTO_TRANSACAO.


## 7. Limpieza y columna auxiliar

In [7]:
df = df.drop(columns=["fee_year_month"])
print(f"Shape final: {df.shape}")
df.isnull().sum()[lambda s: s > 0]

Shape final: (5868722, 20)


Series([], dtype: int64)